**[🏠 Course Home](../README.md)** | **📖 [Conceptual Track](../conceptual/12_bayesian_regression_and_glms.ipynb)** | **🤖 [AI Track](12_bayesian_regression_and_glms.ipynb)** | **🐍 [Python Track](../python/12_bayesian_regression_and_glms.ipynb)** | **🔵 [R Track](../r/12_bayesian_regression_and_glms.ipynb)** | **⬅️ [Prev: Ch 11: Two-Group t-Test](11_t_test_frequentist_vs_bayesian.ipynb)** | **➡️ [Next: Ch 13: Hierarchical Models & Shrinkage](13_hierarchical_models_and_partial_pooling.ipynb)**

*🧭 Part III: Applied Modeling & Hierarchy — Chapter 12 of 22*

---

# Chapter 12: Bayesian Regression & GLMs
### *The Two Uncertainty Ribbons, Centered Geometry, and Aleatoric Guarantees*

---

> **🧭 Course Context & Why This Module Matters in the AI Era:**
>
> In Chapters 1 through 11, we estimated static, isolated parameters: a
> single conversion rate, a fleet-wide sensor location, or the difference
> between two treatment groups. In production systems, variables are rarely
> isolated. Latency depends conditionally on traffic volume (QPS), memory
> utilization scales with concurrent connections, and component failure
> rates accelerate with operating temperature.
>
> When modern engineers ask Large Language Models (LLMs) to construct
> predictive models or certify service-level agreements (SLAs), AI assistants
> suffer from three catastrophic failure modes:
> 1. **The Single-Ribbon Fallacy**: Conflating the uncertainty of the average
>    trend line $\mu(x)$ with the dispersion of individual future requests
>    $y_{\text{new}}$. This mistake leads teams to publish SLAs that fail on
>    over $25\%$ of production requests.
> 2. **Uncentered Geometric Pathology**: Failing to center explanatory variables
>    ($x - \bar{x}$), producing severe covariance ridges between the intercept
>    and slope that induce silent Hamiltonian Monte Carlo (HMC) sampling
>    breakdowns.
> 3. **Linear Link Boundary Violations**: Fitting unconstrained linear
>    equations to bounded probabilities, predicting absurd negative failure
>    rates ($p < 0$) or probabilities exceeding $1.0$.
>
> This module teaches you how to prompt and constrain AI coding agents as
> rigorous probabilistic systems engineers.


## Part 1: The Naive Prompt Trap & Forensic Post-Mortem

### The Prompt That Fools the AI

An infrastructure reliability engineer wants to predict API service latency as
a function of incoming request volume (QPS) to establish a contractual 95%
Service Level Agreement (SLA). They also want to predict the probability of
server memory exhaustion based on CPU temperature.

They submit the following conversational prompt to an advanced AI coding
assistant:

```text
Hey! We have telemetry data showing server latency (y in ms) as a function of
QPS load (x). Fit a Bayesian linear regression in Python to find the 95%
confidence band for SLA provisioning. Also, fit a linear regression on server
temperature to predict whether a server crashes (1) or stays healthy (0).
Provide clean visualization code and explain our 95% latency guarantee at 800
QPS.
```

---

### The Plausible AI Hallucination

The AI enthusiastically generates Python code using PyMC or Stan. The code
defines:

```python
# AI Generated Code (Flawed!)
with pm.Model() as model:
    alpha = pm.Normal("alpha", mu=0, sigma=100)
    beta = pm.Normal("beta", mu=0, sigma=10)
    sigma = pm.HalfNormal("sigma", sigma=20)
    
    mu = alpha + beta * qps_data  # Uncentered!
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=latency_data)
    trace = pm.sample(2000)

# Computing the "95% SLA Band"
qps_grid = np.linspace(100, 1000, 100)
mu_samples = trace.posterior["alpha"].values + trace.posterior["beta"].values * qps_grid[:, None]
sla_lower = np.percentile(mu_samples, 2.5, axis=1)
sla_upper = np.percentile(mu_samples, 97.5, axis=1)
```

The AI provides the following commentary:
> *"At 800 QPS, your server latency is guaranteed with 95% confidence to fall
> between **118.2 ms** and **124.6 ms**. You can safely set your customer SLA
> timeout to **125 ms**! For server crash prediction, our linear regression
> shows that at 45°C, the estimated crash probability is **-0.08** (essentially
> zero)."*

---

### Forensic Post-Mortem: Why the System Crashes

```
     NAIVE AI SLA HORROR: CONFLATING THE TWO UNCERTAINTY RIBBONS
 Latency
  (ms)
   160 |                                          *   *  *  True Outliers
   150 |---------------------------------------*------------- [Real 95% Bound: 150ms]
   140 |                             *  *   *       *  *
   130 |                         *        *    *  *
   125 |===================================================== [AI Claimed SLA: 125ms]
   120 |                     ------------------.--------      [AI Mean Ribbon:
   115 |                    /  Mean Line μ(x)  /               118ms - 124ms]
   100 |         *    *  * -------------------                (Width = 6ms!)
    80 |   *  *    *   *   *
       +---------------------------------------------------
      100           400           700           900  QPS
```

#### 1. Conflating Mean Credible Ribbon with Posterior Predictive Ribbon
The AI plotted the $95\%$ credible interval of the **mean parameter** $\mu(x) =
\mathbb{E}[y \mid x]$. This interval reflects purely **epistemic uncertainty**
(how well the regression line is estimated). Because $N = 100$, the sample
mean is known with tight precision ($\text{width} \approx 6.4\text{ ms}$).

However, an individual request does not arrive at the mean line! It arrives
with residual physical dispersion $\sigma = 15.0\text{ ms}$ (aleatoric
uncertainty):
$$y_{\text{new}} \sim \mathcal{N}\left(\mu(x),\, \sigma^2\right)$$
The variance of a new observation is:
$$\operatorname{Var}(y_{\text{new}} \mid x) = \operatorname{Var}(\mu(x)) + \sigma^2$$
The true $95\%$ **Posterior Predictive Interval** at $800$ QPS spans from
**$91.8\text{ ms}$ to $150.2\text{ ms}$** ($\text{width} \approx 58.4\text{
ms}$). Setting an SLA at $125\text{ms}$ causes **over $25\%$ of customer
requests to breach the contract**, triggering catastrophic outages and penalty
clauses!

#### 2. Uncentered Predictors & Correlated Valleys
By evaluating `mu = alpha + beta * qps`, the intercept $\alpha$ represents
latency at $0\text{ QPS}$ (an extrapolation outside the operational range).
Because the average QPS is $\bar{x} = 550$, tilting the slope $\beta$ forces
$\alpha$ to swing drastically in the opposite direction:
$$\operatorname{Cov}(\hat{\alpha}, \hat{\beta}) = -\bar{x} \operatorname{Var}(\hat{\beta}) \implies \rho(\alpha, \beta) \approx -0.98$$
This creates an ultra-narrow diagonal canyon in parameter space, causing HMC
trajectories to make tiny steps, inflate $\hat{R}$, and drop effective sample
size (ESS) by an order of magnitude.

#### 3. Linear Probability Impossibility
Predicting crash events with an unconstrained linear equation produces
negative probabilities ($p = -0.08$) and probabilities greater than $1.0$ at
high temperatures. A Generalized Linear Model (GLM) with a Logit link function
$\text{logit}(p) = \ln(p / (1-p))$ is required.


## Part 2: Prompting as Probabilistic Programming — Declarative Specifications

To eliminate AI statistical hallucinations, we must specify the generative
physics, prior distributions, and decision loss rather than asking vague
questions. Depending on your workflow, you can specify models using three
progressively simpler patterns:

---

### Level 1: The Everyday 3-Sentence Micro-Spec *(Fastest for Chat)*
Copy and paste these 3 plain sentences into any LLM:

```markdown
1. Data: Server latency vs QPS load; center the predictor x_c = x - x_bar
   to orthogonalize intercept and slope.
2. Dual Bands: Distinguish the narrow 95% Credible Band of the mean line mu(x)
   from the wide 95% Posterior Predictive Band y_new ~ Normal(mu(x), sigma).
3. GLM Link: For binary crash outcomes, use a logistic link logit(p) = alpha + beta*x
   to guarantee valid probabilities in [0, 1].
```

---

### Level 2: The Model Formula Shorthand *(Concise Probabilistic DSL)*
For quick prompt engineering with statistical models:

```text
Linear:   y_i ~ Normal(alpha + beta * (x_i - x_bar), sigma)
Logistic: y_i ~ Bernoulli(logit^{-1}(alpha + beta * T_i))
SLA Band: y_new ~ Normal(mu(x), sigma)  (Posterior Predictive Band)
```

---

### Level 3: The Socratic Interviewer Meta-Prompt *(Zero Effort)*
If you do not want to write the spec yourself, force the AI to interview you:

> *"Act as a Systems Regression Statistician. I am modeling server latency against
traffic load. Ask me 3 questions about predictor centering, mean vs individual
prediction bands, and SLA risk before fitting a regression."*

---

<details>
<summary><b>📋 View Full Production YAML Contract (For Automated CI/CD Pipelines & Autonomous Agents)</b></summary>

```yaml
declarative_probabilistic_specification:
  module: "bayesian_regression_and_glms"
  domain: "server_performance_and_hardware_reliability"
  
  observables:
    continuous_response:
      name: "latency_ms"
      symbol: "y"
      domain: "Real(> 0)"
    binary_response:
      name: "node_crash"
      symbol: "z"
      domain: "Binary {0, 1}"
    predictors:
      load:
        name: "qps"
        symbol: "x"
        centering: "x_c = x - mean(x)"
      thermal:
        name: "temp_celsius"
        symbol: "T"
        centering: "T_c = T - mean(T)"

  linear_regression_model:
    mean_structure: "mu_i = alpha_c + beta * (x_i - x_bar)"
    likelihood: "y_i ~ Normal(mu_i, sigma)"
    priors:
      alpha_c: "Normal(mu=120.0, sigma=20.0)  # Latency at mean QPS"
      beta: "Normal(mu=0.08, sigma=0.04)     # Latency increase per QPS"
      sigma: "Exponential(lam=0.067)          # Expected residual noise ~ 15ms"

  logistic_glm_model:
    link_function: "logit(p_i) = gamma_c + delta * (T_i - T_bar)"
    likelihood: "z_i ~ Bernoulli(p_i)"
    priors:
      gamma_c: "Normal(mu=-3.0, sigma=1.0)   # Baseline crash log-odds at mean temp"
      delta: "Normal(mu=0.15, sigma=0.05)    # Thermal sensitivity coefficient"

  predictive_ribbon_contracts:
    epistemic_band:
      name: "Credible Interval of Mean Line"
      target: "mu(x) = alpha_c + beta * (x - x_bar)"
      percentiles: [2.5, 97.5]
      purpose: "Evaluating scientific parameter knowledge"
    aleatoric_predictive_band:
      name: "Posterior Predictive Interval"
      target: "y_new ~ Normal(mu(x), sigma)"
      percentiles: [2.5, 97.5]
      purpose: "Contractual SLA sizing and infrastructure provisioning"

  decision_contract:
    sla_target_percentile: 0.95
    sla_confidence_level: 0.95
    safe_operating_temperature_rule:
      condition: "P(Crash | T_safe) <= 0.05 at 95% posterior confidence"
```

</details>


## Part 3: The 4 Mandatory Agentic Verification Gates

Before trusting code or conclusions from an AI agent, verify these four gates:

```
        THE 4 MANDATORY AGENTIC VERIFICATION GATES (CH 12)
 ┌───────────────────────┐      ┌─────────────────────────┐
 │ Gate 1: Prior Pred    │ ───► │ Gate 2: SBC Parameter   │
 │ Prior draws yield     │      │ True α, β, σ recovered  │
 │ physical latencies    │      │ within 95% CI in >90%   │
 └───────────────────────┘      └─────────────────────────┘
             │                               │
             ▼                               ▼
 ┌───────────────────────┐      ┌─────────────────────────┐
 │ Gate 3: Centering     │ ───► │ Gate 4: Retrodiction    │
 │ |Corr(α_c, β)| < 0.10 │      │ 93-97% of observations  │
 │ R_hat < 1.01, Div = 0 │      │ inside Predictive Band  │
 └───────────────────────┘      └─────────────────────────┘
```

### Gate 1: Prior Predictive Check
* **Objective**: Ensure the priors do not place mass on physical
  impossibilities.
* **Criterion**: Simulate 1,000 parameter sets from the prior and evaluate
  simulated latency $y_{\text{sim}}$ across load $x \in [100, 1000]$.
  Assert that $P(y_{\text{sim}} < 0) = 0.000$ and
  $P(y_{\text{sim}} > 1000\text{ ms}) < 0.001$. For the logistic model,
  assert that prior failure probabilities span $[0.001, 0.50]$.

### Gate 2: Simulation-Based Calibration (SBC) Fake-Data Recovery
* **Objective**: Guarantee that the inference engine can accurately recover
  known ground-truth values without software bugs.
* **Criterion**: Set $\alpha_{\text{true}} = 120.0$, $\beta_{\text{true}} =
  0.080$, $\sigma_{\text{true}} = 15.0$. Generate synthetic datasets with $N =
  100$. Run MCMC and confirm the 95% posterior credible interval covers the
  true parameters in at least 90 out of 100 simulated trials.

### Gate 3: Centering & Geometric Invariants
* **Objective**: Confirm that parameter correlation has been eliminated.
* **Criterion**: Compute empirical posterior correlation:
  $$|\operatorname{Corr}(\alpha_c, \beta)| < 0.10$$
  Verify sampling health: $\hat{R} < 1.01$ for all parameters, bulk ESS $>
  1000$, tail ESS $> 1000$, and zero divergent transitions.

### Gate 4: Posterior Predictive Retrodiction
* **Objective**: Ensure the observation model captures real aleatoric variance.
* **Criterion**: Evaluate empirical coverage of the dataset:
  $$\text{Coverage} = \frac{1}{N}\sum_{i=1}^N \mathbb{I}\left(y_i \in [y_{2.5\%}(x_i),\, y_{97.5\%}(x_i)]\right)$$
  Empirical coverage must fall strictly between **$93\%$ and $97\%$**.
  If empirical coverage is below $80\%$, the agent mistakenly checked the
  mean line ribbon $\mu(x)$ instead of the predictive distribution
  $y_{\text{new}}$.


## Part 4: Worked Agent Execution Transcript & Telemetry

### Dataset Profile & MCMC Execution
* Sample size: $N = 100$ latency points across loads $x \in [100, 1000]$ QPS.
* Mean load: $\bar{x} = 550.0\text{ QPS}$.
* Sampler: No-U-Turn Sampler (NUTS), 4 chains, 2,000 warmup, 2,000 draws.
* Divergences: $0$. Max $\hat{R} = 1.0002$. Min ESS = $4,821$.

```text
================================================================================
                    MCMC POSTERIOR TELEMETRY LOG (CENTERED)
================================================================================
Parameter      Mean      SD   95% HPD Lower   95% HPD Upper    R-hat    ESS Bulk
--------------------------------------------------------------------------------
alpha_c      120.12    1.51          117.15          123.08   1.0001        5120
beta           0.081   0.005           0.071           0.091   1.0002        4892
sigma         14.88    1.08          12.82           17.04   1.0001        5410
--------------------------------------------------------------------------------
Cov(alpha_c, beta) = -0.0018  ==>  Corr(alpha_c, beta) = -0.024 (Orthogonal!)
================================================================================
```

---

### The Two Uncertainty Ribbons: Quantile Breakdown

| Metric at Load $x$ | 200 QPS ($x_c = -350$) | 550 QPS ($x_c = 0$) | 800 QPS ($x_c = +250$) |
| :--- | :--- | :--- | :--- |
| **Mean Line $\mu(x)$ [Epistemic]** | $91.8 \pm 2.2\text{ ms}$ | $120.1 \pm 1.5\text{ ms}$ | $140.4 \pm 2.0\text{ ms}$ |
| **95% Credible Band of $\mu(x)$** | $[87.5,\, 96.1]\text{ ms}$ | $[117.2,\, 123.1]\text{ ms}$ | $[136.5,\, 144.3]\text{ ms}$ |
| **Mean Line Band Width** | **$8.6\text{ ms}$** | **$5.9\text{ ms}$** | **$7.8\text{ ms}$** |
| **Posterior Predictive $y_{\text{new}}$** | $91.8 \pm 15.0\text{ ms}$ | $120.1 \pm 14.9\text{ ms}$ | $140.4 \pm 15.0\text{ ms}$ |
| **95% Predictive Band for Points**| **$[62.2,\, 121.4]\text{ ms}$** | **$[90.7,\, 149.5]\text{ ms}$** | **$[110.8,\, 170.0]\text{ ms}$** |
| **Predictive Band Width** | **$59.2\text{ ms}$** | **$58.8\text{ ms}$** | **$59.2\text{ ms}$** |

```
                       THE DUAL UNCERTAINTY BEAM
 Latency
  (ms)
   170 |                                             *  [Upper Pred Band: 170ms]
   160 |                                         *   *
   150 |                                 *   *       *
   140 |                             ===.====.========  [Mean Ribbon: 136-144ms]
   130 |                         *   *   *   *
   120 |                 ========.===.========          [Mean Ribbon: 117-123ms]
   110 |             *   *   *   *                      [Lower Pred Band: 111ms]
   100 |     ========.===.========                      [Mean Ribbon: 88-96ms]
    90 | *   *   *   *
    60 | * [Lower Pred Band: 62ms]
       +-------------------------------------------------------------
       100          300           550           800          1000 QPS
```

---

### GLM Failure Rate Analysis: Safe Operating Temperature Envelope
* Dataset: $N = 80$ server runs across temperatures $T \in [40, 90]^\circ\t
  across operational thermal bounds.
* Mean Temperature: $\bar{T} = 65.0^\circ\text{C}$.
* Logit Model: $\text{logit}(p) = \gamma_c + \delta (T - 65.0)$.
* Posterior Estimates: $\gamma_c = -2.85 \pm 0.32$, $\delta = 0.162 \pm 0.021$.

```text
================================================================================
               RELIABILITY ENVELOPE DECISION TELEMETRY
================================================================================
Policy: Failure probability must not exceed 5.0% with 95% posterior confidence.
Operating Target: Find T_safe such that P(p(T_safe) <= 0.05 | Data) >= 0.95.

Median Failure Curve:
  p_median(T) = logit^{-1}(-2.85 + 0.162 * (T - 65.0))
  Crossing at p = 0.05: T = 65.0 + (logit(0.05) - (-2.85)) / 0.162 = 64.4 deg C.

95th Percentile Upper Failure Curve (Conservative Guarantee):
  Evaluated across 4,000 posterior MCMC parameter trajectories.
  Certified Safe Operational Limit: T_safe = 58.7 deg C.
  
Warning: Operating at the point-estimate limit (64.4 deg C) exposes the fleet
to an unhedged 50% probability of exceeding the contractual 5% failure cap!
================================================================================
```


## Part 5: Naive AI Prompt vs. Calibrated Epistemic Specification

| Evaluation Dimension | Naive AI Assistant Output | Calibrated Probabilistic Specification |
| :--- | :--- | :--- |
| **Predictor Geometry** | Raw $x$: $\operatorname{Corr}(\alpha, \beta) \approx -0.98$, slow MCMC convergence | Centered $x - \bar{x}$: $\operatorname{Corr}(\alpha_c, \beta) \approx 0$, orthogonal Hessian |
| **Uncertainty Ribbon** | Plots mean line $\mu(x)$ ($\text{width} \approx 6\text{ms}$); ignores residual noise | Computes both $\mu(x)$ and $y_{\text{new}} \sim \mathcal{N}(\mu, \sigma)$ ($\text{width} \approx 59\text{ms}$) |
| **SLA Guarantee Risk** | Under-provisions SLA threshold by $25\text{ms}$; massive SLA breaches | Fully provisions for individual request dispersion with 95% coverage |
| **Binary Outcomes** | Linear regression; outputs negative failure rates ($p = -0.08$) | Logistic GLM; logit link bounds failure rates strictly in $[0, 1]$ |
| **Sampling Diagnostics** | Ignores divergences and ESS warnings | Enforces $\hat{R} < 1.01$, ESS $> 1000$, and zero divergences |
| **Decision Logic** | Symmetric point-estimate threshold ($p_{\text{mean}} < 0.05$) | Asymmetric 95th percentile upper confidence bound ($T_{\text{safe}}$) |


## Part 6: 🎯 Check Your Understanding — 3-Tier "Audit the AI" Exercises

Test your skills in auditing AI-generated probabilistic regression code.

---

### 🟢 Tier 1: Spot the AI Hallucination
An AI agent presents the following code to compute the "95% latency SLA limit":

```python
# Agent snippet
mu_post = trace.posterior["alpha"] + trace.posterior["beta"] * 900
sla_limit = float(np.percentile(mu_post, 95.0))
print(f"95% of all requests at 900 QPS will complete under {sla_limit:.1f} ms")
```

**Task**: Identify the critical bug in the agent's claim and explain why DevOps
will be paged within minutes of deploying this SLA.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:
The agent computed the 95th percentile of the **posterior distribution of the
mean** $\mu(900)$, not the 95th percentile of the **posterior predictive
distribution** $y_{\text{new}}$.

The parameter $\mu(900)$ represents where the *average of an infinite number of
requests* sits. An individual user request suffers from aleatoric variance
$\sigma^2$. The code completely omitted drawing random normal perturbations
with standard deviation $\sigma$.

Because $\sigma \approx 15\text{ ms}$, approximately $50\%$ of requests at the
mean will exceed $\mu(900)$, and nearly $30\%$ of individual requests will
exceed the agent's `sla_limit`. The correct computation is:

```python
# Calibrated correction
mu_post = trace.posterior["alpha_c"] + trace.posterior["beta"] * (900 - x_bar)
sigma_post = trace.posterior["sigma"]
y_pred = np.random.normal(mu_post, sigma_post)
calibrated_sla = float(np.percentile(y_pred, 95.0))
```
</details>

---

### 🟡 Tier 2: Refactor the Flawed Specification
An engineer drafted this prompt:
> *"Fit a linear model $y = \beta_0 + \beta_1 \cdot \text{mem} + \beta_2 \cdot
> \text{cache\_misses}$ to predict job latency, where memory is in GB
> ($[16, 512]$) and cache misses are in millions ($[1, 100]$)."*

**Task**: Rewrite this into a declarative contract that enforces centering and
computes both epistemic and aleatoric intervals.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:

```yaml
declarative_probabilistic_specification:
  module: "multivariate_job_latency_regression"
  observables:
    response: "completion_time_sec ~ Real(> 0)"
    predictors:
      memory_gb:
        symbol: "m"
        transform: "m_c = m - mean(m)"
      cache_misses_m:
        symbol: "c"
        transform: "c_c = c - mean(c)"
  likelihood: "completion_time_sec ~ Normal(mu, sigma)"
  mean_formula: "mu = alpha_c + beta_m * m_c + beta_c * c_c"
  priors:
    alpha_c: "Normal(mu=60.0, sigma=15.0)  # Completion time at mean covariates"
    beta_m: "Normal(mu=-0.1, sigma=0.05)   # Memory reduction benefit"
    beta_c: "Normal(mu=0.5, sigma=0.2)     # Cache penalty"
    sigma: "Exponential(lam=0.1)           # Residual variance"
  required_outputs:
    - "epistemic_credible_band_mu: percentiles [2.5, 97.5]"
    - "aleatoric_predictive_band_y: percentiles [2.5, 97.5]"
  invariants:
    - "Max absolute correlation among alpha_c, beta_m, beta_c < 0.15"
    - "R_hat < 1.01 for all parameters"
```
</details>

---

### 🔴 Tier 3: Architect an Autonomous Agent Contract
You are designing an autonomous monitoring agent that must certify the maximum
safe operating temperature $T_{\text{max}}$ for a cluster of 500 edge servers.
The safety requirement states that the probability of hardware failure $p(T)$
must not exceed $1\%$ with $99\%$ posterior credibility.

**Task**: Write the complete prompt contract for the agent, including link
function, prior specifications, MCMC sampling thresholds, and decision rule.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:

```yaml
autonomous_agent_contract:
  role: "Cluster Hardware Safety Certification Agent"
  objective: "Determine maximum certified operating temperature T_safe"
  model_specification:
    family: "Bernoulli GLM"
    link: "logit"
    linear_predictor: "eta_i = gamma_c + delta * (T_i - mean(T))"
    priors:
      gamma_c: "Normal(mu=-4.6, sigma=1.0)  # logit(0.01) ~ -4.6"
      delta: "HalfNormal(sigma=0.2)         # Strict positive thermal risk"
  mcmc_invariants:
    target_accept: 0.95
    max_rhat: 1.005
    min_ess_bulk: 2000
    allow_divergences: 0
  decision_rule:
    criterion: "P(logit^{-1}(gamma_c + delta * (T_safe - T_bar)) <= 0.01 | Data) >= 0.99"
    algorithm:
      step_1: "Extract 4,000 posterior parameter draws (gamma_c^{(s)}, delta^{(s)})"
      step_2: "For temperature candidate T in grid [40.0, 95.0, step=0.1]:"
      step_3: "  Compute p^{(s)}(T) = 1 / (1 + exp(-(gamma_c^{(s)} + delta^{(s)} * (T - T_bar))))"
      step_4: "  Compute 99th percentile: p_99(T) = quantile(p^{(s)}(T), 0.99)"
      step_5: "  Find highest T where p_99(T) <= 0.01"
      step_6: "Output T_safe = max(T)"
```
</details>


## Part 7: Key Takeaways & Navigation

### Core Architectural Rules for AI Regression Systems
1. **Never Confuse $\mu(x)$ with $y_{\text{new}}$**: The mean line measures
   scientific epistemic knowledge; individual prediction bands capture the
   reality of production variation. Always size SLAs using posterior predictive
   draws.
2. **Always Center Continuous Predictors ($x - \bar{x}$)**: Centering decouples
   the intercept from slopes, eliminates collinear valleys, and restores
   efficient MCMC sampling.
3. **Use GLMs for Bounded Quantities**: Probabilities live on $[0, 1]$; rates
   live on $(0, \infty)$. Enforce logit or log link functions so the model
   respects physical reality.
4. **Demand the Four Verification Gates**: Before acting on AI recommendations,
   validate prior checks, fake-data recovery, parameter orthogonality, and
   posterior retrodiction coverage.

---

### 🧭 Course Navigation

* 🏠 **[Course Home & Master Syllabus](../README.md)**
* ⬅️ **Previous**: **[Ch 11: Two-Group t-Test (Freq vs. Bayes)](11_t_test_frequentist_vs_bayesian.ipynb)** *(Part II: Comparative Inference)*
* ➡️ **Next**: **[Ch 13: Hierarchical Models & Shrinkage](13_hierarchical_models_and_partial_pooling.ipynb)** *(Part III: Applied Modeling, Regression, Hierarchy & Model Comparison)*
* 🔄 **Companion Tracks for Chapter 12**: [📖 Conceptual](../conceptual/12_bayesian_regression_and_glms.ipynb) | [🤖 AI Track](12_bayesian_regression_and_glms.ipynb) | [🐍 Python](../python/12_bayesian_regression_and_glms.ipynb) | [🔵 R](../r/12_bayesian_regression_and_glms.ipynb)
